# Kustannuskorvausanalyysi

Tässä muistikirjassa demonstroidaan, kuinka luodaan agenteja, jotka käyttävät liitännäisiä käsitelläkseen matkakustannuksia paikallisista kuittikuvista, generoivat kustannuskorvaussähköpostin ja visualisoivat kustannustiedot piirakkakaavion avulla. Agentit valitsevat dynaamisesti funktioita tehtäväkontekstin perusteella.

Vaiheet:
1. OCR-agentti käsittelee paikallisen kuittikuvan ja poimii matkakustannustiedot.
2. Sähköpostianturi luo kustannuskorvaussähköpostin.

### Esimerkki matkakustannusskenaariosta:
Kuvittele, että olet työntekijä, joka matkustaa liiketapaamiseen toiseen kaupunkiin. Yritykselläsi on käytäntö korvata kaikki kohtuulliset matkakustannukset. Tässä erittely mahdollisista matkakustannuksista:
- Kuljetus:
Menopaluu lento kotikaupungistasi kohdekaupunkiin.
Taksi- tai kyytipalvelut lentokentälle ja takaisin.
Paikallinen liikenne kohdekaupungissa (kuten julkinen liikenne, vuokra-autot tai taksit).

- Majoitus:
Kolmen yön hotellimajoitus keskitasoisessa liiketason hotellissa lähellä kokouspaikkaa.

- Ruokailu:
Päivittäinen ruokailutuki aamiaiseen, lounaaseen ja illalliseen yrityksen päivärahan mukaisesti.

- Muut kulut:
Pysäköintimaksut lentokentällä.
Internet-yhteyden maksut hotellissa.
Tipit tai pienet palvelumaksut.

- Dokumentointi:
Toimitat kaikki kuitit (lennot, taksit, hotelli, ruokailut jne.) sekä täytetyn kustannuskorvausraportin korvausta varten.


## Tuo tarvittavat kirjastot

Tuo notebookille tarvittavat kirjastot ja moduulit.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Määritä Kulumallit

 Luo Pydantic-malli yksittäisille kuluillesi ja ExpenseFormatter-luokka, joka muuntaa käyttäjän kyselyn jäsenneltyyn kulutietoon.

 Jokainen kulu esitetään muodossa:
 `{'date': '07-Mar-2025', 'description': 'lento kohteeseen', 'amount': 675.99, 'category': 'Kuljetus'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Työkalujen määrittely - Sähköpostin luominen

Luo työkalutoiminto, joka generoi sähköpostin kulukorvaushakemuksen lähettämistä varten.
- Tämä työkalu käyttää Microsoft Agent Frameworkin `@tool` -koristetta.
- Se laskee kulujen kokonaissumman ja muotoilee tiedot sähköpostin sisällöksi.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Työkalu matkakulujen purkamiseen kuittikuvista

Luo työkalufunktio matkakulujen purkamiseksi kuittikuvista.
- Tämä työkalu käyttää Microsoft Agent Frameworkin `@tool`-koristetta.
- Se lukee kuittikuvan, koodaa sen base64-muotoon ja palauttaa tietojen URI:n agentin analysoitavaksi.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Kulujen käsittely

Määrittele agentit ja kytke ne peräkkäiseen työnkulkuun käyttämällä `WorkflowBuilder`-luokkaa.
- OCR-agentti poimii jäsennellyt kulutiedot kuittikuvasta käyttäen `load_receipt_image`-työkalua.
- Sähköpostiavustaja ottaa poimitut tiedot ja luo ammattimaisen kulukorvausviestin käyttäen `generate_expense_email`-työkalua.
- `WorkflowBuilder` ja `add_edge` luovat peräkkäisen putkiston: OCR-agentti → Sähköpostiavustaja.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Pääfunktio

Rakenna peräkkäinen työnkulku ja suorita se käsittelemään kuittikuva ja luomaan kulukorvausviesti.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty tekoälypohjaisella käännöspalvelulla [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta huomioithan, että automaattikäännöksissä saattaa esiintyä virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä tiedoissa suosittelemme ammattilaisen tekemää ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
